# ARISTA growth–interaction cell-type bubble — CytoBridge API

**Objective.** Recompute per-cell growth and interaction magnitude, aggregate by time and cell type, and redraw the bubble panel using only public `CytoBridge` APIs.


## Plan

1. Load either the portable published checkpoint or a current retrained checkpoint.
2. Compute growth and interaction with the package component API.
3. Aggregate `(time, celltype)` groups and export CSV/SVG plus a runtime manifest.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'downstream_helpers').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from downstream_helpers.arista_api import (
    AristaSpatiotemporalConfig,
    assert_package_only_runtime,
    run_arista_growth_interaction_api,
)
from downstream_helpers.runner import display_svg_outputs

DEVICE = os.environ.get('CYTOBRIDGE_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
SMOKE = os.environ.get('CYTOBRIDGE_SMOKE', '0') == '1'
MODEL_FORMAT = os.environ.get('ARISTA_MODEL_FORMAT', 'legacy')
ALIGNED_H5AD = os.environ.get('ARISTA_ALIGNED_H5AD') or None
MODEL_DIR = os.environ.get('ARISTA_MODEL_DIR') or None
DEVICE, SMOKE, MODEL_FORMAT


## Configuration

Smoke mode limits each timepoint to 256 cells and validates wiring only. Full mode uses every observed cell. Dataset-specific choices stay here; component evaluation, aggregation, and plotting are shared package functions.


In [ ]:
if MODEL_FORMAT == 'current' and (ALIGNED_H5AD is None or MODEL_DIR is None):
    raise ValueError('Current mode requires ARISTA_ALIGNED_H5AD and ARISTA_MODEL_DIR.')

config = AristaSpatiotemporalConfig(
    output_name='arista_growth_interaction_api' + ('_smoke' if SMOKE else ''),
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    model_format=MODEL_FORMAT,
    random_seed=42,
    device=DEVICE,
    run_communication=False,
    run_3d=False,
)
config


In [ ]:
result = run_arista_growth_interaction_api(
    config,
    max_cells_per_timepoint=256 if SMOKE else None,
)
assert_package_only_runtime()
result


## Results

Each dot represents one cell type at one timepoint. Position encodes mean interaction magnitude and mean growth; dot size encodes the number of cells in the group.


In [ ]:
display_svg_outputs([result.bubble_path])
print(result.grouped_csv.read_text(encoding='utf-8')[:3000])
print(result.manifest_path.read_text(encoding='utf-8'))


## Next checks

- Compare published, auto-threshold retrained, and historical-threshold retrained summaries.
- Inspect rare cell types with small dot sizes before biological interpretation.
